In [1]:
from ccdc import io
from ccdc.io import EntryReader

from itertools import islice

from tqdm import tqdm

import pandas as pd
import time

In [3]:
csd = EntryReader('CSD')

### 1. Extract all records to single CSV table

In [ ]:
all_df = []

start = time.time()

total_entry = 500
for entry in tqdm(islice(csd, total_entry), total=total_entry, desc="Processing"):

    # check organic
    is_organic = entry.is_organic

    # check num components
    num_component = len(entry.molecule.components)

    # check metal
    has_metal = any(atom.is_metal for atom in entry.molecule.atoms)

    # add result
    all_df.append({
        "ID":entry.identifier,
        "SMILES": entry.molecule.smiles,
        "IS_ORGANIC": is_organic,
        "HAS_METAL":has_metal,
        "NUM_COMPONENT": num_component,
    })

delta_time = round(time.time() - start, 1)

all_df = pd.DataFrame(all_df)
all_df.to_csv("csd_all.csv", index=False)

print(f"Sequential : Processed {len(all_df)//1000}k entries in {delta_time}s")

In [11]:
all_df

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
1,AACANI10,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
2,AACANI11,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
...,...,...,...,...,...
495,ABAZPT,CC(=O)N1=NC(=[NH][Pt]21[NH]=C(N=N2C(C)=O)C(C)(...,False,True,1
496,ABAZUA,Cc1cc(C)c(c(C)c1)B1(c2ccccc2c2ccc(cn12)c1ccc(c...,True,False,1
497,ABAZUB,O.COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23...,True,False,2
498,ABAZUB01,COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)C...,True,False,2


In [ ]:
from CSDTools.extraction import run_extraction

all_df = run_extraction(output_path="csd_all.csv")

Extraction (24 workers):   2%|▏         | 31500/1436119 [00:19<12:11, 1920.64entry/s]

In [ ]:
all_df

### 2. Select only organic one component molecules

In [ ]:
from CSDTools.processing import filter_entries

all_df = pd.read_csv("csd_all.csv")
df_filtered = filter_entries(all_df)
print(f"Selected {len(df_filtered)} entries")

In [1]:
df_filtered

NameError: name 'df_filtered' is not defined

### 3. Count the number of crystal forms

In [ ]:
from CSDTools.processing import count_forms

df_counted = count_forms(df_filtered)
print(f"Unique molecules: {df_counted['inchi'].nunique()}")

In [8]:
df_counted

,smiles,num_forms,publication_years
0,[Se-][As]1[As]2[As]3[Se][As]4[As]5[As]([Se-])[...,1,1991.0
1,ClB123[As]45[As]61B14(Cl)B25(Cl)B361Cl,1,1995.0
2,IB1234[BH]567[BH]89%10[BH]%11%12%13[BH]158B12%...,1,1991.0
3,Cl[As]12(Cl)[O-][As]3(Cl)(Cl)Cl41[As]1(Cl)(Cl)...,1,2001.0
4,S1[As]2S[As]3[As]1[As]3S2,1,2019.0
...,...,...,...
324980,S=P12SP3(=S)SP(=S)(S1)SP(=S)(S2)S3,2,1998.0|2025.0
324981,S1P2SP3P1P3S2,1,1997.0
324982,S=P12SP3SP(S1)P2S3,1,2025.0
324984,S=P12SP3SP(=S)(SP3S1)S2,1,2025.0


In [ ]:
n_total = df_counted["inchi"].nunique()
n_mono = df_counted.groupby("inchi")["num_forms"].first().eq(1).sum()
n_poly = df_counted.groupby("inchi")["num_forms"].first().gt(1).sum()

print(f"Total molecules: {n_total}")
print(f"Monomorphs: {n_mono} ({round(100 * n_mono / n_total, 1)} %)")
print(f"Polymorphs: {n_poly} ({round(100 * n_poly / n_total, 1)} %)")

In [ ]:
df_per_mol = df_counted.groupby("inchi").first()
false_monomorphs = df_per_mol[df_per_mol["polymorph"].isna() & (df_per_mol["num_forms"] > 1)]
n_mono_db = df_per_mol["polymorph"].isna().sum()

print(f"Molecules marked as monomorphs in DB but polymorphic by InChI: {len(false_monomorphs)}")
print(f"i.e. {round(100 * len(false_monomorphs) / n_mono_db, 1)}% of DB monomorphs")

In [12]:
df_counted.to_csv("csd_counted.csv", index=False)

### 4. Properties

In [9]:
entry = csd[0]
properties = dir(entry.crystal)
properties

['Contact',
 'Disorder',
 'HBond',
 'MillerIndices',
 'PeriodicBondChain',
 'ReducedCell',
 'Void',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_chemical_info',
 '_crystal',
 '_crystal_info',
 '_csv',
 '_decode_inclusion',
 '_identifier',
 '_voids_volume_telemetry',
 'add_hydrogens',
 'are_atoms_symmetry_related',
 'assign_bonds',
 'asymmetric_unit_molecule',
 'atoms_on_special_positions',
 'calculate_voids',
 'calculated_density',
 'cell_angles',
 'cell_lengths',
 'cell_volume',
 'centre_molecule',
 'contact_network',
 'contacts',
 'copy',
 'crystal_system',
 'disorder',
 'disordered_molecule',
 'formula',
 'fractional_to_orthogonal',
 'from_stri

In [ ]:
BANNED_LIST = ['melting_point_display_string', 'input_melting_point_range', 'formatted_melting_point_text', 'formatted_melting_point_range' ]
MAX_ENTRIES_TEST = 100

prop_list = {}
for prop in dir(entry):
    if prop.startswith('_') or prop in BANNED_LIST:
        continue

    try:
        attr_class = getattr(type(entry), prop, None)
        if isinstance(attr_class, property):
            doc = getattr(attr_class.fget, '__doc__', None)
        else:
            doc = getattr(getattr(entry, prop), '__doc__', None)

        doc_clean = doc.split('>>>')[0].split(':')[0].strip() if doc else None

        example_value = None
        for i, test_entry in enumerate(islice(csd, MAX_ENTRIES_TEST)):
            try:
                value = getattr(test_entry, prop)
                if value:
                    example_value = value
                    break
            except:
                continue

        # if str(example_value)[0] == '<':
        #     continue

        prop_list[prop] = {
            "description": doc_clean,
            "example": str(example_value) if example_value is not None else "No example found"
        }

    except Exception as e:
        if "Solubility Platform" not in str(e):
            print(f"Error with {prop} : {e}")

for prop, infos in prop_list.items():
    if infos['example'][0] == "<":
        print(f"--- {prop} ---")
        print(f"Description : {infos['description']}")
        print(f"Exemple : {infos['example']}")
        print()

print(len(prop_list))

In [ ]:
BANNED_LIST = ['melting_point_display_string', 'input_melting_point_range',
               'formatted_melting_point_text', 'formatted_melting_point_range']
MAX_ENTRIES_TEST = 100

entry = csd[0]
molecule = entry.molecule

prop_list = {}
for prop in dir(molecule):
    if prop.startswith('_') or prop in BANNED_LIST:
        continue

    try:
        attr_class = getattr(type(molecule), prop, None)
        if isinstance(attr_class, property):
            doc = getattr(attr_class.fget, '__doc__', None)
        else:
            doc = getattr(getattr(molecule, prop), '__doc__', None)

        doc_clean = doc.split('>>>')[0].split(':')[0].strip() if doc else None

        example_value = None
        for test_entry in islice(csd, MAX_ENTRIES_TEST):
            try:
                value = getattr(test_entry.molecule, prop)
                if value:
                    example_value = value
                    break
            except:
                continue

        # if str(example_value)[0] == '<':
        #     continue
        
        prop_list[prop] = {
            "description": doc_clean,
            "example": str(example_value) if example_value is not None else "No example found"
        }

    except Exception as e:
        if "Solubility Platform" not in str(e):
            print(f"Error with {prop} : {e}")

for prop, infos in prop_list.items():
    print(f"--- {prop} ---")
    print(f"Description : {infos['description']}")
    print(f"Exemple : {infos['example']}")
    print()

print(len(prop_list))